In [ ]:
import torch
import sys
import types

sys.path.insert(0, "/inwdata2a/sudhanshu/Unet_training_script")

# Import ALL classes from unet_wrapper.py
from models.backbones.unet_wrapper import (
    UnetDec,
    UnetEnc,
    Unet,
    UNetBackbone,
    UnetUpSampleDec,
    UnetUpSample,
    UnetUpSample_modified
)

# Create fake modules
fake_top    = types.ModuleType("eye_internal_segmentor")
fake_model  = types.ModuleType("eye_internal_segmentor.model")
fake_unet   = types.ModuleType("eye_internal_segmentor.model.unet_wrapper")

# Map every class the .pt file might reference
fake_unet.UnetDec               = UnetDec
fake_unet.UnetEnc               = UnetEnc
fake_unet.Unet                  = Unet
fake_unet.UNetBackbone          = UNetBackbone
fake_unet.UnetUpSampleDec       = UnetUpSampleDec
fake_unet.UnetUpSample          = UnetUpSample
fake_unet.UnetUpSample_modified = UnetUpSample_modified

sys.modules["eye_internal_segmentor"]                    = fake_top
sys.modules["eye_internal_segmentor.model"]              = fake_model
sys.modules["eye_internal_segmentor.model.unet_wrapper"] = fake_unet

# Load
model = torch.load(
    "/inwdata2a/sudhanshu/landmarks_only_training/outputs_landmarks_unet/dyotana-model.pt",
    map_location="cpu",
    weights_only=False
)

print("Type:", type(model))

for k in model:
    print(k)


Type: <class 'dict'>
cfg
model


In [4]:
# See what keys are in the dict
print("Keys in checkpoint:", list(model.keys()))

# Peek at the type of each value
for k, v in model.items():
    print(f"  {k}: {type(v)}")

Keys in checkpoint: ['cfg', 'model']
  cfg: <class 'yacs.config.CfgNode'>
  model: <class 'models.backbones.unet_wrapper.UnetUpSample'>


## Dyotana Eye Segmentation Model (`dyotana-model.pt`)

**Saved on:** 31 Jul 2024  
**Author:** Dyotana

---

### Model Contents
- **Config:** All hyperparameters and training details as a config dictionary
- **Model:** Full model object (not just weights), requiring the original Python classes to load

---

### Training & Data
- **Input:** `64×64` grayscale eye images (single-channel)
- **Task:** Segmentation of `pupil` + `iris` + `sclera` + 1 blink channel  
  → **5 output channels**
- **Epochs:** 50
- **Device:** `cuda:3`
- **Activation:** LeakyReLU  
- **Dataset:** Internal lab dataset (no eye-close data)

---

### Architecture
- **Type:** `UnetUpSample` (**pre-modified version**)
    - *Important*: No sigmoid on `output2`
- **Output:**
    - `seg2.weight` shape: `[5, 8, 1, 1]` (**5 output channels**)
    - Full encoder & decoder:
      ```
      enc64 → enc32 → enc16 → enc8 → conv → dec8 → dec16 → dec32 → dec64 → seg1 → seg2
      ```

---

### Loading Notes
- Since the **full model object** was saved (not just weights), you **must have the exact code** for `UnetUpSample` (from the older version) and all dependencies in your Python path to reload this model.
- If you see missing class definition errors, add the corresponding model scripts or use `add_safe_globals` for safe deserialization.

---

**In summary:**  
This checkpoint holds a gray-scale, 5-class, old-architecture U-Net model for internal eye part segmentation, with config and model structure included for replicability and inference—**but requires the exact original model code to load**.

In [4]:
# The actual model is inside the "model" key
actual_model = model["model"]
cfg          = model["cfg"]

print("Model type:", type(actual_model))
print("Config:", cfg)

print("\n--- State Dict Keys & Shapes ---")
for k, v in actual_model.state_dict().items():
    print(f"  {k:40s}  {str(v.shape)}")

# Re-save cleanly
torch.save(
    actual_model.state_dict(),
    "/inwdata2a/sudhanshu/landmarks_only_training/outputs_landmarks_unet/dyotana-model-fixed.pt"
)
print("\n✅ Re-saved as state_dict at dyotana-model-fixed.pt")

Model type: <class 'models.backbones.unet_wrapper.UnetUpSample'>
Config: accum_iter: 256
blink_loss_scale: 4000
blinkpath: /home/ubuntu/inwdata1/dyotanad/old_internalLab_CVAT_data_processed_v3_final/deeplabv3_ckpt35_with_blinkModel/dms_prod_tight_64x64/labels_o
datapath: /home/ubuntu/inwdata1/dyotanad/old_internalLab_CVAT_data_processed_v3_final/deeplabv3_ckpt35_with_blinkModel/dms_prod_tight_64x64/data/SEED1
devicename: cuda:3
do_eye_close: False
imagepath: /home/ubuntu/inwdata1/dyotanad/old_internalLab_CVAT_data_processed_v3_final/deeplabv3_ckpt35_with_blinkModel/dms_prod_tight_64x64/inputs
is_test: False
labelpath: /home/ubuntu/inwdata1/dyotanad/old_internalLab_CVAT_data_processed_v3_final/deeplabv3_ckpt35_with_blinkModel/dms_prod_tight_64x64/labels_jointmap
labelpath2: /home/ubuntu/inwdata1/dyotanad/old_internalLab_CVAT_data_processed_v3_final/deeplabv3_ckpt35_with_blinkModel/dms_prod_tight_64x64/labels_jointmap2
logdir: /home/ubuntu/inwdata/dyotanad/eye_segmentation/code/eye_inter